In [2]:
!pip install transformers datasets accelerate -q

In [3]:
import torch, math
from transformers import (GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed)
from datasets import Dataset

set_seed(42)

In [4]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        out = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [6]:
review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

baseline = {}

print("=== BEFORE FINE-TUNING ===")
for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}\nOutput: {baseline[p]}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BEFORE FINE-TUNING ===

Prompt: This product is
Output: This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most

Prompt: I bought this phone and
Output: I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I have been using phones that have the Snapdragon 616 processor with the 8

Pro

In [7]:
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments perfectly.",
    "the sound quality of these headphones is incredible with deep bass.",
    "this smartwatch tracks my steps accurately and is very reliable.",
    "great wireless earbuds with noise cancellation.",
    "excellent value for money and premium build quality.",
    "highly recommend this product for daily use.",
    "this camera takes stunning photos in low light.",
    "easy to set up and works perfectly.",
    "best purchase i made this year."
]

dataset = Dataset.from_dict({'text': corpus})

In [8]:
tokenized = dataset.map(lambda x: tokenizer(
    x['text'],
    truncation=True,
    padding='max_length',
    max_length=128
), batched=True, remove_columns=['text'])

split = tokenized.train_test_split(test_size=0.2)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [9]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./gpt2-reviews',
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy='no'
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.190492
20,1.483230
30,0.766370
40,0.652174


TrainOutput(global_step=40, training_loss=1.5230663895606995, metrics={'train_runtime': 7.178, 'train_samples_per_second': 11.145, 'train_steps_per_second': 5.573, 'total_flos': 5225840640000.0, 'train_loss': 1.5230663895606995, 'epoch': 10.0})

In [11]:
print("\n=== AFTER FINE-TUNING ===")

for p in review_prompts:
    output = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Before: {baseline[p][:100]}")
    print(f"After:  {output[:100]}")


=== AFTER FINE-TUNING ===

Prompt: This product is
Before: This product is made from high quality, lightweight stainless steel. If you are looking for somethin
After:  This product is a blast to set up and works perfectly. The quality of the silicone absorbs fingerpri

Prompt: I bought this phone and
Before: I bought this phone and I have not used it on a lot of people. I have also not used it on any other 
After:  I bought this phone and it handles all my calls perfectly. I recommend this phone to anyone looking 

Prompt: The quality of this item
Before: The quality of this item in the item description (and if the item is already in stock) will determin
After:  The quality of this item is outstanding for the money. I bought this laptop for college and it handl


In [12]:
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
recipes = [
    "to make butter chicken marinate chicken in yogurt and spices.",
    "heat butter and cook onions until golden brown.",
    "add tomato puree and cook for ten minutes.",
    "cook chicken until tender and add cream.",
    "serve hot with naan or rice.",
    "boil pasta and reserve pasta water.",
    "fry pancetta until crispy.",
    "mix eggs and cheese for sauce.",
    "combine pasta with sauce and serve.",
    "add vegetables and stir fry on high heat."
]

dataset2 = Dataset.from_dict({'text': recipes})

In [14]:
tok2 = dataset2.map(lambda x: tokenizer2(
    x['text'],
    truncation=True,
    padding='max_length',
    max_length=128
), batched=True, remove_columns=['text'])

split2 = tok2.train_test_split(test_size=0.2)

collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

trainer2 = Trainer(
    model=model2,
    args=training_args,
    train_dataset=split2['train'],
    eval_dataset=split2['test'],
    data_collator=collator2
)

trainer2.train()

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Step,Training Loss
10,3.183808
20,1.719056
30,1.045688
40,0.795178


TrainOutput(global_step=40, training_loss=1.6859324336051942, metrics={'train_runtime': 3.8353, 'train_samples_per_second': 20.859, 'train_steps_per_second': 10.429, 'total_flos': 5225840640000.0, 'train_loss': 1.6859324336051942, 'epoch': 10.0})

In [15]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta",
    "To prepare a dish"
]

print("\n=== RECIPE OUTPUT ===")

for p in recipe_prompts:
    print(f"\nPrompt: {p}")
    print(generate_text(model2, tokenizer2, p))


=== RECIPE OUTPUT ===

Prompt: To make butter chicken
To make butter chicken marinate chicken in yogurt and spices. Add cream and serve.

To serve, add cream and cook on high heat. Serve hot.

Adapted from Vegan Chicken Salad with Chicken and Cheese. Print Recipe Notes Nutrition Information Yield: 7 servings

Calories: 4.3g

Fat: 0g

Saturated fat: 0g

Cholesterol: 191mg

Sodium: 9g

Carbohydrates:

Prompt: For pasta
For pasta with marinate and cook rice for ten minutes until golden brown. Serve with rice or noodles.

How to Make Dijun Curry with Chicken Curry

Pour hot sesame oil in yogurt and serve. Serve with rice, curry powder and serve.

How to Make Dijun Chicken Curry with Chicken

Chickpeas or chicken breast for dipping. Add curry powder and serve.

How to Make Dijun Chicken Curry with Chicken

Sa

Prompt: To prepare a dish
To prepare a dish. Heat marinate pasta in yogurt and spices. Add chicken and cook for ten minutes. Serve.

This post may contain links to Amazon or other pa